In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

# 1. Load engineered datasets
X_train = pd.read_csv('../data/processed/X_train_engineered.csv')
X_test = pd.read_csv('../data/processed/X_test_engineered.csv')
y_train = pd.read_csv('../data/processed/y_train.csv')['SalePrice_log'].values

print(f"Loaded X_train shape: {X_train.shape}")
print(f"Loaded X_test shape: {X_test.shape}")
print(f"Loaded y_train shape: {y_train.shape}")

Loaded X_train shape: (1458, 306)
Loaded X_test shape: (1459, 306)
Loaded y_train shape: (1458,)


In [2]:
# --- Setup 10-Fold Cross-Validation ---
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# --- Ridge Baseline Model ---
oof_ridge = np.zeros(len(X_train))
preds_ridge = np.zeros(len(X_test))
ridge_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]
    
    model = Ridge(alpha=10.0, random_state=42)
    model.fit(X_tr, y_tr)
    
    val_preds = model.predict(X_va)
    oof_ridge[val_idx] = val_preds
    preds_ridge += model.predict(X_test) / kf.n_splits
    
    score = root_mean_squared_error(y_va, val_preds)
    ridge_scores.append(score)

cv_ridge = np.mean(ridge_scores)
print(f"Ridge 10-Fold Mean CV RMSE: {cv_ridge:.5f}")

Ridge 10-Fold Mean CV RMSE: 0.11217


In [3]:
# --- Random Forest Baseline Model ---
oof_rf = np.zeros(len(X_train))
preds_rf = np.zeros(len(X_test))
rf_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]
    
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)
    
    val_preds = model.predict(X_va)
    oof_rf[val_idx] = val_preds
    preds_rf += model.predict(X_test) / kf.n_splits
    
    score = root_mean_squared_error(y_va, val_preds)
    rf_scores.append(score)

cv_rf = np.mean(rf_scores)
print(f"Random Forest 10-Fold Mean CV RMSE: {cv_rf:.5f}")

Random Forest 10-Fold Mean CV RMSE: 0.13236


In [4]:
# --- Create Experiment Record Table ---
results_df = pd.DataFrame({
    'Model': ['Ridge Baseline', 'RandomForest Baseline'],
    'CV RMSE': [cv_ridge, cv_rf]
})

print(results_df)

# Save experiment records to CSV
results_df.to_csv('../data/processed/experiment_log.csv', index=False)

                   Model   CV RMSE
0         Ridge Baseline  0.112174
1  RandomForest Baseline  0.132360
